# Day 23 Tutorial — Fabric Lakehouse

**Goal:** Medallion in Lakehouse, shortcuts, Delta tables.


## Concepts
- Lakehouse = files + Delta tables + Spark/SQL
- Shortcuts avoid full data copies
- Medallion bronze/silver/gold tables


### Environment setup
Skip pip install on Databricks/Fabric. Locally you may need: `pip install pyspark pandas`.


In [ ]:
# %pip install pyspark==3.5.1 pandas -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark


In [ ]:
bronze = spark.createDataFrame(
    [(1, 'c1', '100', '2024-01-01'), (2, 'c1', 'x', '2024-01-01'), (3, 'c2', '50', '2024-01-02')],
    ['order_id', 'customer_id', 'amount_str', 'dt'],
)
silver = bronze.filter(F.col('amount_str').rlike(r'^\d+(\.\d+)?$')).withColumn(
    'amount', F.col('amount_str').cast('double')
)
gold = silver.groupBy('dt').agg(F.sum('amount').alias('revenue'))
silver.show()
gold.show()
